# Modele sklearn

## Wstępna konfiguracja
---

### Importowanie bibliotek

In [11]:
import pandas as pd

from mlxtend.data import loadlocal_mnist

from sklearn import model_selection
from sklearn import metrics

from sklearn import tree
from sklearn import ensemble
from sklearn import neural_network

### Constants initialization

In [12]:
class config:
    
    # Data files paths
    DOWNLOADED_IMAGES_PATH = '../data/t10k-images.idx3-ubyte'
    DOWNLOADED_LABELS_PATH = '../data/t10k-labels.idx1-ubyte'

    # Format data config
    RANDOMIZE_DATA = True

    # General project settings
    FOLDS_CNT = 5

## Konfiguracja danych

### Wczytanie danych

In [13]:
# Load data from files
X, y = loadlocal_mnist(
    images_path=config.DOWNLOADED_IMAGES_PATH,
    labels_path=config.DOWNLOADED_LABELS_PATH
)

# Generate names of columns
pixel_columns = [f"pixel{i}" for i in range(len(X[0]))]

# Create pandas dataframe
df = pd.DataFrame(X, columns=pixel_columns)

# Add label column
df["label"] = y

# Show data
df

,pixel0,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,label
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,7
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,2
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,2
9996,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,3
9997,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,4
9998,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,5


### Podział danych na foldy

<img src="./images/image1.png" alt="image1" width="1300"/>
<!-- ![image1](./images/image1.png) -->
<!-- ![image1](https://towardsdatascience.com/wp-content/uploads/2023/12/1N45hocCMP0u4nXLe0WuSvw.png) -->

In [14]:
# Creating kfold column
df["kfold"] = -1

# Dividing data into segments and eventually randoming data
folds_model = model_selection.StratifiedKFold(
    n_splits=config.FOLDS_CNT,
    shuffle=config.RANDOMIZE_DATA
)
for fold, (train, test) in enumerate(folds_model.split(df, df["label"].values)):
    df.loc[test, "kfold"] = fold
    
    print(f"{fold}. {train}, {test}")
    
# df

0. [   0    2    3 ... 9997 9998 9999], [   1    4    9 ... 9984 9986 9987]
1. [   1    2    4 ... 9997 9998 9999], [   0    3    5 ... 9992 9993 9994]
2. [   0    1    2 ... 9997 9998 9999], [   8   20   23 ... 9977 9982 9985]
3. [   0    1    3 ... 9993 9994 9997], [   2    7   27 ... 9996 9998 9999]
4. [   0    1    2 ... 9996 9998 9999], [   6   12   14 ... 9981 9988 9997]


## Trenowanie modelu

### Dyspozytor modelu

In [15]:
class model_dispatcher:
    
    model_names = [
        "decision_tree_gini",
        "decision_tree_entropy",
        "random_forest",
        # "neural_network"
    ]

    models = {
        "decision_tree_gini": tree.DecisionTreeClassifier(
            criterion="gini"
        ),
        "decision_tree_entropy": tree.DecisionTreeClassifier(
            criterion="entropy"
        ),
        "random_forest": ensemble.RandomForestClassifier(),
        "neural_network": neural_network.MLPClassifier()
    }

### Trenowanie oraz testowanie modelu

In [16]:
def use_model(df_train, df_test, metrics_method):
    
    # Data configuration
    x_train = df_train.loc[:, df_train.columns != "label"].values
    y_train = df_train.loc[:, "label"].values

    x_test = df_test.loc[:, df_test.columns != "label"].values
    y_test = df_test.loc[:, "label"].values
    
    # Fit model
    model.fit(x_train, y_train)

    # Predict results
    predicted_data = model.predict(x_test)
    
    # Calculate and return metrics scores
    scores = metrics_method(
        y_true=y_test,
        y_pred=predicted_data
    )
    return scores

In [17]:
# Initialize dictionary with metrics scores
metrics_dict = {}

# Loop through all models
for model_name in model_dispatcher.model_names:
    model = model_dispatcher.models[model_name]
    
    # Initialize new element in metrics dictionary
    metrics_dict[model_name] = []

    # Test single model on different folds
    for fold in range(config.FOLDS_CNT):
        
        # Initialize train dataframe and test dataframe
        df_train = df.loc[df["kfold"] != fold, df.columns != "kfold"]
        df_test = df.loc[df["kfold"] == fold, df.columns != "kfold"]

        # Calculate, add to dictionary and print accuracy score
        score = use_model(df_train, df_test, metrics.accuracy_score)
        metrics_dict[model_name].append(score)
        
        print(f"{model_name}: {fold} - {score:.3f}")

decision_tree_gini: 0 - 0.803
decision_tree_gini: 1 - 0.798
decision_tree_gini: 2 - 0.804
decision_tree_gini: 3 - 0.800
decision_tree_gini: 4 - 0.796
decision_tree_entropy: 0 - 0.811
decision_tree_entropy: 1 - 0.811
decision_tree_entropy: 2 - 0.811
decision_tree_entropy: 3 - 0.791
decision_tree_entropy: 4 - 0.822
random_forest: 0 - 0.948
random_forest: 1 - 0.953
random_forest: 2 - 0.956
random_forest: 3 - 0.955
random_forest: 4 - 0.952


## Metryki

### Słownik metryk

In [8]:
metrics_dict

{'decision_tree_gini': [0.804, 0.798, 0.8045, 0.822, 0.805],
 'decision_tree_entropy': [0.793, 0.834, 0.8135, 0.8245, 0.811],
 'random_forest': [0.949, 0.94, 0.958, 0.966, 0.948]}

### Metryki w DataFrame-ach

In [9]:
metrics_df = pd.DataFrame(metrics_dict)

display(metrics_df)
display(metrics_df.mean().to_frame().transpose().rename(index={0: "avg"}))

,decision_tree_gini,decision_tree_entropy,random_forest
0,0.8040,0.7930,0.949
1,0.7980,0.8340,0.940
2,0.8045,0.8135,0.958
3,0.8220,0.8245,0.966
4,0.8050,0.8110,0.948


,decision_tree_gini,decision_tree_entropy,random_forest
avg,0.8067,0.8152,0.9522


In [10]:
metrics_df_tran = pd.DataFrame(metrics_dict).transpose()

display(metrics_df_tran)
display(metrics_df_tran.mean(axis=1).to_frame().rename(columns={0: "avg"}))

,0,1,2,3,4
decision_tree_gini,0.804,0.798,0.8045,0.8220,0.805
decision_tree_entropy,0.793,0.834,0.8135,0.8245,0.811
random_forest,0.949,0.940,0.9580,0.9660,0.948


,avg
decision_tree_gini,0.8067
decision_tree_entropy,0.8152
random_forest,0.9522
